# Client-Product Annual Behavior Analysis
This notebook allows you to explore how specific clients purchase specific products year over year.

In [4]:
import pandas as pd
import numpy as np
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

# Load Data
df = pd.read_csv('data/master_commodities_clean.csv', low_memory=False)
df['Fecha'] = pd.to_datetime(df['Fecha'])
df['Año'] = df['Fecha'].dt.year

print(f'Dataset loaded with {len(df)} rows.')

Dataset loaded with 91978 rows.


## 1. Data Aggregation
We aggregate the data by Client, Product, and Year.

In [5]:
df_annual = df.groupby(['Id_Cliente', 'Id_Producto', 'Año']).agg({
    'Unidades': 'sum',
    'Valores_H': 'sum',
    'Num.Fact': 'nunique'
}).reset_index()
df_annual.columns = ['Id_Cliente', 'Id_Producto', 'Año', 'Total_Unidades', 'Total_Valor', 'Num_Pedidos']

# Get lists for selectors
clients = sorted(df_annual['Id_Cliente'].unique())

print('Aggregation complete.')

Aggregation complete.


## 2. Interactive Visualization
Use the dropdowns to select a Client and a Product.

In [6]:
client_dropdown = widgets.Dropdown(options=clients, description='Client ID:')
product_dropdown = widgets.Dropdown(description='Product ID:')

def update_products(*args):
    client_id = client_dropdown.value
    products = sorted(df_annual[df_annual['Id_Cliente'] == client_id]['Id_Producto'].unique())
    product_dropdown.options = products

client_dropdown.observe(update_products, 'value')
update_products() # Initialize

@widgets.interact(client_id=client_dropdown, product_id=product_dropdown)
def plot_behavior(client_id, product_id):
    # 1. Filter detailed daily data
    detailed_subset = df[(df['Id_Cliente'] == client_id) & (df['Id_Producto'] == product_id)].copy()
    
    if detailed_subset.empty:
        print('No data found for this selection.')
        return
    
    # 2. Prepare aligned X-axis for yearly comparison
    # We use a dummy year (2000) to align the months and days across all years
    detailed_subset['Fecha_Aliniada'] = detailed_subset['Fecha'].apply(lambda x: x.replace(year=2000))
    detailed_subset['Año_Str'] = detailed_subset['Año'].astype(str)
    
    # 3. Faceted Bar Chart (One below the other per year)
    # This allows comparing the same dates across different years easily
    fig = px.bar(detailed_subset.sort_values('Fecha'), 
                 x='Fecha_Aliniada', 
                 y='Unidades', 
                 facet_row='Año_Str',
                 color='Año_Str',
                 title=f'Daily Activity by Year (Comparative): Client {client_id} - Product {product_id}',
                 hover_data={'Fecha': '|%d %B %Y', 'Unidades': True, 'Num.Fact': True, 'Fecha_Aliniada': False, 'Año_Str': False},
                 labels={'Unidades': 'Units', 'Fecha_Aliniada': 'Month', 'Año_Str': 'Year'},
                 category_orders={"Año_Str": sorted(detailed_subset['Año_Str'].unique(), reverse=True)})
    
    # Adjust layout for better readability
    fig.update_layout(
        height=250 * len(detailed_subset['Año'].unique()), 
        showlegend=False,
        margin=dict(t=50, b=50, l=50, r=50)
    )
    
    # Fix X-axis to show months correctly and align all plots
    fig.update_xaxes(tickformat='%b', dtick='M1', title='Time of Year')
    
    # Clean up facet labels
    fig.for_each_annotation(lambda a: a.update(text=f"Year {a.text.split('=')[-1]}"))
    
    fig.show()
    
    # 4. Annual summary table
    subset_annual = df_annual[(df_annual['Id_Cliente'] == client_id) & (df_annual['Id_Producto'] == product_id)]
    print("\nAnnual Summary:")
    display(subset_annual[['Año', 'Total_Unidades', 'Total_Valor', 'Num_Pedidos']].sort_values('Año'))

interactive(children=(Dropdown(description='Client ID:', options=(np.int64(26), np.int64(30), np.int64(46), np…